In [ ]:
# =========================================
# 1. Configuración
# =========================================

CLEAN_FILE = "transactions_clean.parquet"
BUCKET_NAME = "your-gcs-bucket"  # <-- cambia por tu bucket real
BLOB_NAME = "transactions_clean.parquet"

EXPECTED_COLUMNS = [
    "transaction_id",
    "customer_id",
    "transaction_amount",
    "transaction_date",
    "payment_method",
    "product_category",
    "quantity",
    "customer_age",
    "customer_location",
    "device_used",
    "ip_address",
    "shipping_address",
    "billing_address",
    "is_fraudulent",
    "account_age_days",
    "transaction_hour"
]


In [ ]:
# =========================================
# 2. Subir archivo (desde tu computadora)
# =========================================

from google.colab import files

uploaded = files.upload()

# Obtener automáticamente el nombre del archivo
raw_file = next(iter(uploaded.keys()))
print(f"Archivo cargado: {raw_file}")

In [ ]:
# =========================================
# 3. Carga del dataset
# =========================================

import pandas as pd

df = pd.read_csv(
    raw_file,
    sep=",",
    quotechar='"',
    engine="python",
    on_bad_lines="skip"
)

print("Shape inicial:", df.shape)
print("Columnas originales:", df.columns.tolist())
df.head()

In [ ]:
# =========================================
# 4. Estandarización de columnas
# =========================================

df.columns = EXPECTED_COLUMNS

print("Columnas estandarizadas:")
print(df.columns.tolist())


In [ ]:
# =========================================
# 5. Limpieza de datos
# =========================================

# Eliminar duplicados
df = df.drop_duplicates()

# Convertir columnas numéricas
numeric_cols = [
    "transaction_amount",
    "quantity",
    "customer_age",
    "is_fraudulent",
    "account_age_days",
    "transaction_hour"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Eliminar filas con valores críticos nulos
required_cols = [
    "transaction_id",
    "customer_id",
    "transaction_amount",
    "transaction_date",
    "payment_method",
    "product_category",
    "quantity",
    "customer_age",
    "customer_location",
    "device_used",
    "is_fraudulent",
    "account_age_days",
    "transaction_hour"
]

df = df.dropna(subset=required_cols)

# Filtrar valores inválidos
df = df[
    (df["transaction_amount"] > 0) &
    (df["quantity"] > 0) &
    (df["customer_age"] > 0) &
    (df["account_age_days"] >= 0) &
    (df["transaction_hour"].between(0, 23))
].copy()

print("Shape después de limpieza:", df.shape)
df.head()

In [ ]:
# =========================================
# 6. Exportar a Parquet
# =========================================

df.to_parquet(CLEAN_FILE, index=False)
print(f"Archivo exportado: {CLEAN_FILE}")


In [ ]:
# =========================================
# 7. Autenticación en GCP
# =========================================

from google.colab import auth
auth.authenticate_user()

# =========================================
# 8. Instalar cliente de Cloud Storage
# =========================================

!pip install -q google-cloud-storage

# =========================================
# 9. Subir archivo a Cloud Storage
# =========================================

from google.cloud import storage

client = storage.Client()
bucket = client.bucket(BUCKET_NAME)

blob = bucket.blob(BLOB_NAME)
blob.upload_from_filename(CLEAN_FILE)

print(f"Archivo subido a: gs://{BUCKET_NAME}/{BLOB_NAME}")